# Modal Component

> Tabbed modal with progress, logs, and resources tabs.

In [ ]:
#| default_exp components.modal

In [ ]:
#| export
from typing import Optional, List

from fasthtml.common import Div, H3, Button, Input, Form, Span, Dialog, Script, FT

from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_action, modal_backdrop
from cjm_fasthtml_daisyui.components.actions.button import btn_modifiers
from cjm_fasthtml_daisyui.components.navigation.tabs import tab, tabs, tabs_styles, tab_content
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, border_dui, text_dui
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, justify
from cjm_fasthtml_tailwind.utilities.layout import display_tw, position, right, top
from cjm_fasthtml_tailwind.core.base import combine_classes

# Design system recipes (V1 button roles)
from cjm_fasthtml_design_system.buttons import buttons

from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig, ResourceSnapshot
from cjm_fasthtml_job_monitor.components.tabs.progress_tab import render_progress_tab
from cjm_fasthtml_job_monitor.components.tabs.logs_tab import render_logs_tab
from cjm_fasthtml_job_monitor.components.tabs.resources_tab import render_resources_tab

## SSE Helpers

The SSE connection element is a hidden div with `hx-ext="sse"` and `sse-connect`
pointing to the streaming endpoint. HTMX opens an `EventSource` to that URL and
processes incoming `sse_message()` payloads as HTML with OOB swaps.

The tab content OOB helpers are used by both the initial modal render and the
SSE stream to build consistent update payloads.

In [ ]:
#| export
_TERMINAL_STATUSES = {'completed', 'failed', 'cancelled'}

def render_sse_connection(
    ids: JobMonitorHtmlIds,       # Element IDs
    urls: JobMonitorUrls,         # Route URLs (progress URL used for SSE endpoint)
    job_id: str,                  # Active job ID (passed as query param)
    is_active: bool = True,       # Whether SSE should be connected
) -> FT:  # Hidden SSE connection div
    """Render the hidden SSE connection element.
    
    When `is_active`, includes hx-ext=sse and sse-connect for real-time updates.
    When not active, renders as an inert hidden div (no connection).
    """
    attrs = {"id": ids.sse_connection, "style": "display:none;"}
    if is_active and job_id:
        attrs["hx_ext"] = "sse"
        attrs["sse_connect"] = f"{urls.progress}?job_id={job_id}"
        attrs["sse_swap"] = "message"
        attrs["hx_swap"] = "none"
    return Div(**attrs)


def render_tab_content_oob(
    ids: JobMonitorHtmlIds,                      # Element IDs
    status: str = 'pending',                     # Job status
    progress_value: float = 0.0,                 # 0.0 to 1.0
    status_message: str = '',                    # Stage message
    started_at: Optional[float] = None,          # Unix timestamp
    completed_at: Optional[float] = None,        # Unix timestamp
    logs: str = '',                              # Log text
    resources: Optional[ResourceSnapshot] = None, # Resource data
) -> tuple:  # (progress_div, logs_div, resources_div) with OOB attrs
    """Render the three tab inner content divs as OOB swap targets."""
    progress_div = Div(
        render_progress_tab(
            ids, status=status, progress_value=progress_value,
            status_message=status_message, started_at=started_at,
            completed_at=completed_at,
        ),
        id=ids.tab_progress,
    )
    progress_div.attrs['hx-swap-oob'] = "true"

    logs_div = Div(
        render_logs_tab(ids, logs=logs),
        id=ids.tab_logs,
    )
    logs_div.attrs['hx-swap-oob'] = "true"

    resources_div = Div(
        render_resources_tab(resources),
        id=ids.tab_resources,
    )
    resources_div.attrs['hx-swap-oob'] = "true"

    return progress_div, logs_div, resources_div


def render_footer_oob(
    ids: JobMonitorHtmlIds,    # Element IDs
    urls: JobMonitorUrls,      # Route URLs
    is_active: bool = True,    # Whether job is active (show cancel)
) -> FT:  # Footer div with OOB attr
    """Render the modal footer with cancel button as OOB swap."""
    children = []
    if is_active:
        children.append(
            Button(
                "Cancel",
                hx_post=urls.cancel,
                hx_swap="none",
                cls=buttons.destructive_cancellable,
            )
        )
    footer = Div(
        *children,
        id=ids.modal_footer,
        cls=combine_classes(flex_display, justify.end, m.t(4)),
    )
    footer.attrs['hx-swap-oob'] = "true"
    return footer


def _make_tab_oob(ids, tab_id, render_fn, *args, **kwargs):
    """Render a single tab's inner content as OOB swap target."""
    div = Div(render_fn(*args, **kwargs), id=tab_id)
    div.attrs['hx-swap-oob'] = "true"
    return div


def render_sse_response(
    ids: JobMonitorHtmlIds,                      # Element IDs
    urls: JobMonitorUrls,                        # Route URLs
    status: str = 'pending',                     # Job status
    progress_value: float = 0.0,                 # 0.0 to 1.0
    status_message: str = '',                    # Stage message
    started_at: Optional[float] = None,          # Unix timestamp
    completed_at: Optional[float] = None,        # Unix timestamp
    logs: Optional[str] = None,                  # Log text (None = skip logs tab update)
    resources: Optional[ResourceSnapshot] = None, # Resource data (None = skip resources tab update)
    include_footer: bool = False,                # Include footer OOB (only needed on state transitions)
    extra_oob: Optional[List[FT]] = None,        # Additional OOB elements (cleanup, callbacks)
) -> FT:  # Div wrapping all OOB elements for sse_message()
    """Build the OOB update payload for an SSE push.
    
    Selectively includes only changed elements:
    - Progress tab: always included (status/progress/elapsed change frequently)
    - Logs tab: included only when `logs` is not None
    - Resources tab: included only when `resources` is not None
    - Footer: included only when `include_footer` is True (state transitions)
    """
    is_active = status not in _TERMINAL_STATUSES
    children = []

    # Progress tab — always
    children.append(_make_tab_oob(
        ids, ids.tab_progress, render_progress_tab,
        ids, status=status, progress_value=progress_value,
        status_message=status_message, started_at=started_at,
        completed_at=completed_at,
    ))

    # Logs tab — only if logs provided
    if logs is not None:
        children.append(_make_tab_oob(
            ids, ids.tab_logs, render_logs_tab,
            ids, logs=logs,
        ))

    # Resources tab — only if resources provided
    if resources is not None:
        children.append(_make_tab_oob(
            ids, ids.tab_resources, render_resources_tab,
            resources,
        ))

    # Footer — only on state transitions
    if include_footer:
        children.append(render_footer_oob(ids, urls, is_active=is_active))

    if extra_oob:
        children.extend(extra_oob)

    return Div(*children)


def get_sse_headers() -> List[FT]:
    """Return the HTMX SSE extension script + cleanup script for app headers.
    
    Consumers should add these to their FastHTML app's hdrs list.
    The SSE extension must load after the main HTMX script.
    """
    return [
        Script(src="https://unpkg.com/htmx-ext-sse@2.2.2"),
        Script("window.addEventListener('beforeunload',()=>{"
               "document.querySelectorAll('[sse-connect]').forEach(e=>{"
               "const d=e['htmx-internal-data'];if(d&&d.sseEventSource)d.sseEventSource.close();"
               "})});"),
    ]

## render_job_modal

Full modal dialog with header, static tab structure, inner content divs,
SSE connection element, and backdrop. The tab structure is rendered once and
never replaced. Only the inner content of each tab and the footer are updated
via OOB swaps pushed through the SSE stream.

In [ ]:
#| export
def render_job_modal(
    config: JobMonitorConfig,                    # Display config
    ids: JobMonitorHtmlIds,                      # Element IDs
    urls: JobMonitorUrls,                        # Route URLs
    job_id: str = '',                            # Active job ID for SSE connection
    status: str = 'pending',                     # Job status
    progress_value: float = 0.0,                 # 0.0 to 1.0
    status_message: str = '',                    # Stage message
    started_at: Optional[float] = None,          # Unix timestamp
    completed_at: Optional[float] = None,        # Unix timestamp
    logs: str = '',                              # Log text
    resources: Optional[ResourceSnapshot] = None, # Resource data
    open_on_render: bool = False,                # Auto-open via JS
) -> FT:  # Dialog element
    """Render the full tabbed modal dialog.
    
    The tab structure (radio inputs + tab-content wrappers) is static.
    Each tab-content wrapper contains an inner div with a stable ID
    that gets OOB-swapped by the SSE stream. This prevents the
    selected tab from resetting on each update.
    
    Closable via: Escape key, X button (top-right), or clicking backdrop.
    """
    is_active = status not in _TERMINAL_STATUSES
    tab_group = f"{ids.prefix}-modal-tabs"

    # Initial tab inner content
    progress_inner = render_progress_tab(
        ids, status=status, progress_value=progress_value,
        status_message=status_message, started_at=started_at,
        completed_at=completed_at,
    )
    logs_inner = render_logs_tab(ids, logs=logs)
    resources_inner = render_resources_tab(resources)

    # Tab structure — radio inputs and tab-content wrappers are static
    tab_children = [
        # Progress tab
        Input(type="radio", name=tab_group, aria_label="Progress",
              checked="checked", cls=str(tab)),
        Div(
            Div(progress_inner, id=ids.tab_progress),
            cls=combine_classes(tab_content, border_dui.base_300, bg_dui.base_100),
        ),
        # Logs tab
        Input(type="radio", name=tab_group, aria_label="Logs", cls=str(tab)),
        Div(
            Div(logs_inner, id=ids.tab_logs),
            cls=combine_classes(tab_content, border_dui.base_300, bg_dui.base_100),
        ),
        # Resources tab
        Input(type="radio", name=tab_group, aria_label="Resources", cls=str(tab)),
        Div(
            Div(resources_inner, id=ids.tab_resources),
            cls=combine_classes(tab_content, border_dui.base_300, bg_dui.base_100),
        ),
    ]

    tab_container = Div(
        *tab_children,
        cls=combine_classes(tabs, tabs_styles.border, w.full),
    )

    # Footer with cancel button
    footer_children = []
    if is_active:
        footer_children.append(
            Button(
                "Cancel",
                hx_post=urls.cancel,
                hx_swap="none",
                cls=buttons.destructive_cancellable,
            )
        )
    footer = Div(
        *footer_children,
        id=ids.modal_footer,
        cls=combine_classes(flex_display, justify.end, m.t(4)),
    )

    # SSE connection (hidden, drives real-time updates)
    sse_el = render_sse_connection(ids, urls, job_id, is_active=is_active)

    # X close button (top-right corner)
    close_btn = Form(
        Button(
            "✕",  # Unicode multiplication sign
            cls=combine_classes(
                buttons.soft_dismissal, btn_modifiers.circle,
                position.absolute, right._2, top._2,
            ),
        ),
        method="dialog",
    )

    # Open script
    open_script = Script(
        f"document.getElementById('{ids.modal}').showModal();"
    ) if open_on_render else Span()

    return Dialog(
        Div(
            # X close button
            close_btn,
            # Header
            H3(config.modal_title, cls=combine_classes(font_weight.bold, font_size.lg)),
            # Tab structure (static)
            tab_container,
            # Footer (OOB-swappable)
            footer,
            # SSE connection (hidden)
            sse_el,
            cls=combine_classes(modal_box, max_w._2xl, w.full),
        ),
        # Backdrop — plain unstyled button per DaisyUI v5 pattern
        Form(Button("close"), method="dialog", cls=str(modal_backdrop)),
        open_script,
        id=ids.modal,
        cls=str(modal),
    )

In [ ]:
# Test render_sse_response — selective updates
from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig, ResourceSnapshot

ids = JobMonitorHtmlIds(prefix="jm")
urls = JobMonitorUrls(trigger="/fa/trigger", progress="/fa/progress", cancel="/fa/cancel")
config = JobMonitorConfig(modal_title="Force Alignment")

# Progress-only (typical running cycle — no logs change, no resources, no footer)
result_min = render_sse_response(
    ids, urls, status='running', progress_value=0.5,
    status_message='Aligning...',
)
child_ids = [c.attrs.get('id') for c in result_min.children if hasattr(c, 'attrs') and 'id' in c.attrs]
assert 'jm-tab-progress' in child_ids
assert 'jm-tab-logs' not in child_ids
assert 'jm-tab-resources' not in child_ids
assert 'jm-modal-footer' not in child_ids
print(f"render_sse_response (progress-only): {len(result_min.children)} children — OK")

# With all tabs and footer (terminal state)
snap = ResourceSnapshot(worker_pid=123, cpu_percent=50.0, memory_rss_mb=1000.0)
result_full = render_sse_response(
    ids, urls, status='completed', progress_value=1.0,
    logs="final logs", resources=snap, include_footer=True,
)
child_ids_full = [c.attrs.get('id') for c in result_full.children if hasattr(c, 'attrs') and 'id' in c.attrs]
assert 'jm-tab-progress' in child_ids_full
assert 'jm-tab-logs' in child_ids_full
assert 'jm-tab-resources' in child_ids_full
assert 'jm-modal-footer' in child_ids_full
print(f"render_sse_response (all tabs + footer): {len(result_full.children)} children — OK")

# Footer excluded when not requested (running state)
result_no_footer = render_sse_response(
    ids, urls, status='running', progress_value=0.5,
    logs="some logs", resources=snap,
)
child_ids_nf = [c.attrs.get('id') for c in result_no_footer.children if hasattr(c, 'attrs') and 'id' in c.attrs]
assert 'jm-modal-footer' not in child_ids_nf
print("render_sse_response (footer excluded): OK")

# With extra OOB
from fasthtml.common import Div as _Div
extra = [_Div("cleanup", id="extra-el")]
result_extra = render_sse_response(
    ids, urls, status='completed', progress_value=1.0,
    logs="final logs", resources=snap, include_footer=True, extra_oob=extra,
)
assert any(c.attrs.get('id') == 'extra-el' for c in result_extra.children if hasattr(c, 'attrs'))
print("render_sse_response (with extra OOB): OK")

In [ ]:
# Test render_job_modal
modal_el = render_job_modal(
    config, ids, urls, job_id="test-123", status='running', progress_value=0.3,
    status_message='Loading model...', open_on_render=True,
)
assert modal_el.tag == 'dialog'
assert modal_el.attrs['id'] == 'jm-modal'
assert 'modal' in modal_el.attrs['class']

# Verify static tab structure inside modal_box
from fasthtml.common import to_xml
html = to_xml(modal_el)
# Tab radio inputs should be present
assert 'name="jm-modal-tabs"' in html
assert 'aria-label="Progress"' in html
assert 'aria-label="Logs"' in html
assert 'aria-label="Resources"' in html
# Inner content divs with stable IDs
assert 'id="jm-tab-progress"' in html
assert 'id="jm-tab-logs"' in html
assert 'id="jm-tab-resources"' in html
# SSE connection element
assert 'id="jm-sse"' in html
assert 'hx-ext="sse"' in html
assert 'sse-connect="/fa/progress?job_id=test-123"' in html
# Footer
assert 'id="jm-modal-footer"' in html
# Open script
assert 'showModal()' in html
print("render_job_modal: OK")

# Test SSE connection is inert when status is terminal
modal_done = render_job_modal(
    config, ids, urls, job_id="test-123", status='completed',
)
html_done = to_xml(modal_done)
assert 'sse-connect' not in html_done
print("render_job_modal (completed): SSE connection inert")

# Test get_sse_headers
hdrs = get_sse_headers()
assert len(hdrs) == 2
assert 'htmx-ext-sse' in hdrs[0].attrs.get('src', '')
print("get_sse_headers: OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()